# Финальный проект. Этап 1 — Загрузка и очистка данных

**Проект:** анализ CRM-данных онлайн-школы программирования X.

**Цель этапа:** загрузить исходные данные, выполнить очистку
в соответствии с условиями проекта и сохранить чистые датасеты в `data_clean/`.

**Принципы работы:**
- исходные файлы в `data/` не изменяются — работаем с копией в памяти;
- каждое решение по очистке объясняется прямо перед кодом;
- результат этапа: 4 чистых CSV + итоговая таблица «было/стало».

**План этапа:**
1. Загрузка данных и первый взгляд
2. Обзор пропусков
3. Очистка Contacts
4. Очистка Calls
5. Очистка Spend
6. Очистка Deals (главная таблица)
7. Итоги и сохранение

In [94]:
# Импортируем библиотеки для работы с данными и визуализации
import datetime as dt
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Настройки отображения: показываем все колонки широких таблиц
pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (10, 5)

In [95]:
# Пути к данным: исходники лежат в data/, результат сохраним в data_clean/
DATA_DIR = Path('../data')
CLEAN_DIR = Path('../data_clean')

# Создаём папку для чистых данных, если её ещё нет
CLEAN_DIR.mkdir(exist_ok=True)

## 1. Загрузка данных и первый взгляд

**Важный момент перед загрузкой.** Колонки с идентификаторами (`Id`,
`Contact Name`, `CONTACTID`) — это 19-значные числа вида
`5805028000000805001`. Тип `float64` хранит точно только ~15–16 значащих
цифр, поэтому если читать Id как число, последние цифры обнулятся и
разные сделки «склеятся» в ложные дубликаты.

**Решение:** все идентификаторы читаем как строки (`dtype=str`).

In [96]:
# Загружаем 4 таблицы из Excel.
# Все колонки-идентификаторы читаем строками
deals = pd.read_excel(
    DATA_DIR / 'Deals (Done).xlsx',
    dtype={'Id': str, 'Contact Name': str},
)
calls = pd.read_excel(
    DATA_DIR / 'Calls (Done).xlsx',
    dtype={'Id': str, 'CONTACTID': str},
)
contacts = pd.read_excel(DATA_DIR / 'Contacts (Done).xlsx', dtype={'Id': str})
spend = pd.read_excel(DATA_DIR / 'Spend (Done).xlsx')

In [97]:
# Фиксируем исходные размеры таблиц —
# в конце этапа сравним их с результатом («было/стало»)
rows_before = {
    'deals': len(deals),
    'calls': len(calls),
    'contacts': len(contacts),
    'spend': len(spend),
}

for name, df in [('Deals', deals), ('Calls', calls),
                 ('Contacts', contacts), ('Spend', spend)]:
    print(f'{name}: {df.shape[0]} строк, {df.shape[1]} колонок')

Deals: 21595 строк, 23 колонок
Calls: 95874 строк, 11 колонок
Contacts: 18548 строк, 4 колонок
Spend: 20779 строк, 8 колонок


### Deals

In [98]:
deals

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,Term,Source,Payment Type,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
0,5805028000055525099,Eva Kent,NaN,B - Medium,Waiting For Payment,NaN,/webinar,webinar1906,"4 days, 21:19:46",NaN,invitation,Webinar,NaN,Web Developer,Morning,15.06.2024 13:50,6.0,NaN,100,"€ 2.900,00",5805028000055474262,"Poland , Gdansk , Al. Grunwaldzka 7, ap. 1a",NaN
1,5805028000053717506,Ben Hall,NaN,C - Low,Call Delayed,NaN,/direct,blog2_DE,01:48:36,NaN,30_05_2024,SMM,NaN,Web Developer,Morning,06.06.2024 14:53,6.0,NaN,3000,"€ 2.900,00",5805028000053715483,NaN,NaN
2,5805028000052825311,Eva Kent,15.06.2024,C - Low,Payment Done,NaN,/direct,blog2_DE,NaN,NaN,30_05_2024,SMM,One Payment,UX/UI Design,Morning,31.05.2024 18:08,11.0,2.0,"€ 3.500,00",3500,5805028000052833288,Novi Sad,NaN
3,5805028000053249254,Quincy Vincent,NaN,C - Low,Qualificated,NaN,eng/web-developer,performancemax_eng_DE,13:03:58,_{region_name}_,NaN,Google Ads,NaN,Web Developer,Morning,03.06.2024 21:51,6.0,NaN,NaN,"€ 2.900,00",5805028000053276436,NaN,NaN
4,5805028000053098851,Ulysses Adams,NaN,C - Low,Qualificated,NaN,/eng,youtube_shorts_DE,05:03:32,bloggersvideo10,Com_august,Youtube Ads,NaN,Web Developer,Morning,03.06.2024 06:05,6.0,NaN,0,"€ 2.900,00",5805028000053116462,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21590,5805028000000948010,Jane Smith,29.08.2023,B - Medium,Lost,needs time to think,eng/digital-marketing,03.07.23women,NaN,b3,women,Facebook Ads,NaN,NaN,NaN,04.07.2023 07:10,NaN,NaN,NaN,NaN,5805028000000979006,NaN,NaN
21591,5805028000000945016,Jane Smith,29.08.2023,A - High,Lost,Changed Decision,eng/digital-marketing,02.07.23wide_DE,"56 days, 19:01:59",b3,wide,Facebook Ads,NaN,NaN,NaN,03.07.2023 20:39,NaN,NaN,NaN,NaN,5805028000000968001,NaN,NaN
21592,5805028000000927004,Bob Brown,09.07.2023,D - Non Target,Lost,Does not speak English,eng/digital-marketing,03.07.23women,NaN,b3,women,Facebook Ads,NaN,NaN,NaN,03.07.2023 20:17,NaN,NaN,NaN,NaN,5805028000000961001,NaN,NaN
21593,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [99]:
deals.describe().round(1)

,Course duration,Months of study
count,3587.0,840.0
mean,10.2,5.4
std,1.8,2.9
min,6.0,0.0
25%,11.0,3.0
50%,11.0,5.0
75%,11.0,8.0
max,11.0,11.0


**Deals — выводы первичного осмотра:**

- info(): из 21 595 строк только **4 165 имеют Initial Amount Paid** — платёж фиксируется лишь у оплативших лидов, остальные пропуски структурные (норма воронки). SLA заполнен у 15 533 строк — у остальных менеджер не ответил или заявка не назначена.
- describe(): **SLA max = 26 908 464 сек (~311 дней)** — грубый выброс, ответ через год. Из-за таких выбросов среднее SLA (~32 ч) сильно искажено, медиана (~5.5 ч) точнее. IAP min = 0, max = 11 500 при mean = 950 — видно, что есть символические оплаты (0, 1, 9 €) и реальные платежи в одном поле: нужен отдельный флаг.

### Calls

In [100]:
calls

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Dialled Number,Outgoing Call Status,Scheduled in CRM,Tag
0,5805028000021593632,15.12.2023 19:05,Sam Young,5805028000021404645,Outbound,7625.0,Attended Dialled,NaN,Completed,0.0,NaN
1,5805028000049344069,10.05.2024 17:34,Victor Barnes,5805028000047779175,Outbound,7200.0,Attended Dialled,NaN,Completed,0.0,NaN
2,5805028000037508725,14.03.2024 19:46,John Doe,5805028000021062552,Inbound,6798.0,Received,NaN,NaN,NaN,NaN
3,5805028000021415097,14.12.2023 15:49,Sam Young,5805028000021263687,Outbound,6368.0,Attended Dialled,NaN,Completed,0.0,NaN
4,5805028000043303087,11.04.2024 16:00,Charlie Davis,5805028000041617254,Outbound,6304.0,Attended Dialled,NaN,Completed,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
95869,5805028000053636072,05.06.2024 17:00,Victor Barnes,NaN,Outbound,NaN,Overdue,NaN,Overdue,1.0,NaN
95870,5805028000053651109,06.06.2024 07:00,Victor Barnes,NaN,Outbound,NaN,Cancelled,NaN,Cancelled,1.0,NaN
95871,5805028000055092068,14.06.2024 16:00,Victor Barnes,NaN,Outbound,NaN,Scheduled,NaN,Scheduled,1.0,NaN
95872,5805028000055626456,17.06.2024 10:00,Victor Barnes,NaN,Outbound,NaN,Scheduled,NaN,Scheduled,1.0,NaN


In [101]:
calls.describe().round(1)

,Call Duration (in seconds),Dialled Number,Scheduled in CRM,Tag
count,95791.0,0.0,86875.0,0.0
mean,165.0,NaN,0.0,NaN
std,401.4,NaN,0.0,NaN
min,0.0,NaN,0.0,NaN
25%,4.0,NaN,0.0,NaN
50%,8.0,NaN,0.0,NaN
75%,98.0,NaN,0.0,NaN
max,7625.0,NaN,1.0,NaN


**Calls — выводы первичного осмотра:**

- info(): **Dialled Number и Tag пусты на 100%** — колонки без данных, удаляем. Call Duration имеет пропуски — у неотвеченных звонков длительность не фиксируется.
- describe(): Call Duration min = 0 — есть звонки с нулевой длительностью (набрали и сбросили). Scheduled in CRM хранится как float вместо bool/int — артефакт Excel.

### Contacts

In [102]:
contacts

,Id,Contact Owner Name,Created Time,Modified Time
0,5805028000000645014,Rachel White,27.06.2023 11:28,22.12.2023 13:34
1,5805028000000872003,Charlie Davis,03.07.2023 11:31,21.05.2024 10:23
2,5805028000000889001,Bob Brown,02.07.2023 22:37,21.12.2023 13:17
3,5805028000000907006,Bob Brown,03.07.2023 05:44,29.12.2023 15:20
4,5805028000000939010,Nina Scott,04.07.2023 10:11,16.04.2024 16:14
...,...,...,...,...
18543,5805028000056889209,Ulysses Adams,21.06.2024 12:11,21.06.2024 14:11
18544,5805028000056889351,Eva Kent,21.06.2024 13:32,21.06.2024 15:32
18545,5805028000056892018,Eva Kent,21.06.2024 10:21,21.06.2024 12:21
18546,5805028000056892055,Yara Edwards,21.06.2024 10:22,21.06.2024 12:23


In [103]:
contacts.describe().round(1)

,Id,Contact Owner Name,Created Time,Modified Time
count,18548,18548,18548,18548
unique,18548,28,17921,16580
top,5805028000000645014,Charlie Davis,10.06.2024 09:00,13.06.2024 17:08
freq,1,2018,13,25


**Contacts — выводы первичного осмотра:**

- info(): все 4 колонки **заполнены на 100%**, 0 пропусков — самая чистая таблица из четырёх.
- describe(): Created Time и Modified Time показаны как object (строки) вместо datetime — нужно привести к типу datetime перед анализом.

### Spend

In [104]:
spend

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,NaN,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,NaN,NaN
2,2023-12-23,Bloggers,NaN,15001,774.00,164,NaN,NaN
3,2023-10-10,Bloggers,NaN,9797,773.00,257,NaN,NaN
4,2024-02-29,Bloggers,NaN,10369,750.00,181,NaN,NaN
...,...,...,...,...,...,...,...,...
20774,2024-06-21,Facebook Ads,17.03.24wide_AT,7,0.07,0,wide,bloggersvideo16com_at
20775,2024-06-21,Tiktok Ads,12.07.2023wide_DE,61,0.16,0,wide,bloggersvideo14com
20776,2024-06-21,Partnership,NaN,0,0.00,0,NaN,NaN
20777,2024-06-21,Tiktok Ads,NaN,0,0.00,0,NaN,NaN


In [105]:
spend.describe().round(1)

,Date,Impressions,Spend,Clicks
count,20779,20779.0,20779.0,20779.0
mean,2024-01-14 22:32:40.864334080,2458.2,7.2,24.0
min,2023-07-03 00:00:00,0.0,0.0,0.0
25%,2023-10-13 00:00:00,0.0,0.0,0.0
50%,2024-01-27 00:00:00,63.0,0.6,1.0
75%,2024-04-16 00:00:00,709.0,5.8,12.0
max,2024-06-21 00:00:00,431445.0,774.0,2415.0
std,NaN,11442.5,26.8,85.2


**Spend — выводы первичного осмотра:**

- info(): Campaign, AdGroup, Ad имеют ~5 000+ пропусков — это **норма для органических каналов** (Organic, Bloggers) без рекламных кампаний, не ошибка.
- describe(): Spend min = 0.0 — есть строки с нулевыми расходами. Визуально заметны **повторяющиеся строки** — нужна проверка на дубликаты: если в дублях нет денег, удаление безопасно, но это надо проверить перед удалением.

## 2. Обзор качества данных

Прежде чем что-то исправлять, нужно **увидеть** проблему и **зафиксировать** её.
В этом разделе по каждой таблице проверяем:
- 2.1 пропущенные значения (NaN)
- 2.2 дубликаты строк
- 2.3 типы данных

По итогам — 2.4 конкретный план очистки.

### 2.1 Пропущенные значения (NaN)

#### Deals

In [106]:
# Считаем долю пропусков в каждой колонке Deals (в процентах)
missing_pct = (deals.isna().mean() * 100).sort_values()
missing_pct.round(1)

Id                      0.0
Stage                   0.0
Page                    0.0
Source                  0.0
Created Time            0.0
Deal Owner Name         0.1
Contact Name            0.3
Quality                10.4
Lost Reason            25.3
Campaign               25.6
SLA                    28.1
Closing Date           32.2
Content                34.5
Term                   42.3
Offer Total Amount     80.6
Initial Amount Paid    80.7
Product                83.4
Course duration        83.4
Education Type         84.7
City                   88.4
Level of Deutsch       94.2
Months of study        96.1
Payment Type           97.7
dtype: float64

In [107]:
[↑ к Deals в разделе 1](#sec-1-deals)  
# Визуализируем пропуски, чтобы оценить картину одним взглядом
ax = missing_pct.plot.barh(figsize=(9, 7), color='steelblue')
ax.set_title('Deals: доля пропусков по колонкам, %')
ax.set_xlabel('% пропусков')
plt.tight_layout() # автоматически подгоняет отступы внутри фигуры, чтобы элементы графика не наезжали друг на друга и не обрезались
plt.show()

SyntaxError: invalid character '↑' (U+2191) (1656126990.py, line 1)

**Вывод по графику:** колонки нижней части (Id, Stage, Source, даты)
почти полные — это «скелет» сделки. Верхняя часть (Payment Type, Months
of study, Level of Deutsch — 90%+ пропусков) заполняется только на
поздних стадиях воронки. Стратегия: чистим ошибки, структурные пропуски
сохраняем как есть.

#### Calls

In [ ]:
# Пропуски по всем колонкам Calls (0 = колонка чистая)
n = len(calls)
na = calls.isna().sum().sort_values(ascending=False)
na_df = na.to_frame("Пропусков")
na_df["% от строк"] = (na_df["Пропусков"] / n * 100).round(1)
na_df

,Пропусков,% от строк
Tag,95874,100.0
Dialled Number,95874,100.0
Scheduled in CRM,8999,9.4
Outgoing Call Status,8999,9.4
CONTACTID,3933,4.1
Call Duration (in seconds),83,0.1
Id,0,0.0
Call Owner Name,0,0.0
Call Start Time,0,0.0
Call Status,0,0.0


**Что нашли в Calls:**

- `Dialled Number` и `Tag` — пустые на **100%** (из 95 874 строк — ни одного значения).
  Эти колонки не несут информации → удалим на этапе очистки.
- `CONTACTID` пустой в ~4% строк — звонки без привязки к контакту; оставляем как есть.
- `Call Duration` и другие пропуски незначительны (< 0.1%).

#### Contacts

In [ ]:
# Пропуски по всем колонкам Contacts (0 = колонка чистая)
n = len(contacts)
na = contacts.isna().sum().sort_values(ascending=False)
na_df = na.to_frame("Пропусков")
na_df["% от строк"] = (na_df["Пропусков"] / n * 100).round(1)
na_df

,Пропусков,% от строк
Id,0,0.0
Contact Owner Name,0,0.0
Created Time,0,0.0
Modified Time,0,0.0


**Что нашли в Contacts:**

Пропусков нет совсем. Таблица контактов — самая чистая из четырёх.
На этапе очистки нужно будет только привести типы дат.

#### Spend

In [ ]:
# Пропуски по всем колонкам Spend (0 = колонка чистая)
n = len(spend)
na = spend.isna().sum().sort_values(ascending=False)
na_df = na.to_frame("Пропусков")
na_df["% от строк"] = (na_df["Пропусков"] / n * 100).round(1)
na_df

,Пропусков,% от строк
AdGroup,6828,32.9
Ad,6828,32.9
Campaign,5994,28.8
Date,0,0.0
Impressions,0,0.0
Source,0,0.0
Clicks,0,0.0
Spend,0,0.0


**Что нашли в Spend:**

- `Campaign`, `AdGroup`, `Ad` — пропуски в ~33% строк.
  По контексту: строки с нулевыми расходами (Bloggers-канал) могут не иметь
  названия кампании — это не ошибка, а особенность выгрузки.
- Критичных пропусков в денежных (`Spend`) и метрических (`Impressions`, `Clicks`) колонках нет.

### 2.2 Дубликаты

Проверяем каждую таблицу отдельно: ищем **полные** дубликаты строк
и дубликаты по ключевому идентификатору.

#### Deals

In [ ]:
# Дубликаты в Deals
full_dup = deals.duplicated().sum()
id_dup = deals["Id"].duplicated().sum()
print(f"Полных дублей строк : {full_dup}")
print(f"Дублей по колонке Id: {id_dup}")

Полных дублей строк : 0
Дублей по колонке Id: 1


**Что нашли в Deals:**

- Полных дублей строк: **0**.
- Дублей по колонке `Id`: **1** — одна сделка присутствует дважды
  с разным содержимым (не полный дубль, а расхождение данных).

Разберём эту строку в разделе 4.1 при очистке Deals.

#### Calls

In [ ]:
# Дубликаты в Calls
full_dup = calls.duplicated().sum()
id_dup = calls["Id"].duplicated().sum()
contact_id_dup = calls["CONTACTID"].duplicated().sum()
print(f"Полных дублей строк : {full_dup}")
print(f"Дублей по колонке Id: {id_dup}")
print(f"Дублей по колонке ContactId: {contact_id_dup}")

Полных дублей строк : 0
Дублей по колонке Id: 0
Дублей по колонке ContactId: 80659


**Что нашли в Calls:**

- Полных дублей строк: **0**.
- Дублей по `CONTACTID` повторяется (см. вывод выше) — это **нормально**.
  Один контакт может иметь десятки звонков, поэтому CONTACTID
 — это не дубликат, а связь "один-ко-многим".

Проблем с дублями в Calls нет.

#### Contacts

In [ ]:
# Дубликаты в Contacts
full_dup = contacts.duplicated().sum()
id_dup = contacts["Id"].duplicated().sum()
print(f"Полных дублей строк : {full_dup}")
print(f"Дублей по колонке Id: {id_dup}")

Полных дублей строк : 0
Дублей по колонке Id: 0


**Что нашли в Contacts:**

- Полных дублей строк: **0**.
- Дублей по `Id`: **0** — каждый контакт уникален.

Таблица чистая по дублям.

#### Spend

In [ ]:
# Дубликаты в Spend
# Id нет — проверяем только полные дубликаты строк
full_dup = spend.duplicated().sum()
print(f"Полных дублей строк: {full_dup}")

# Прежде чем удалять — проверяем, есть ли в дублях деньги
# Если сумма расходов в дублях = 0, удаление безопасно
dup_mask = spend.duplicated(keep=False)
print(f"Сумма Spend в дублях: {spend.loc[dup_mask, 'Spend'].sum():.2f}")

Полных дублей строк: 917
Сумма Spend в дублях: 0.00


**Что нашли:** 917 полных дублей, но сумма расходов в них = 0.
Это строки канала Bloggers без кампании и без трат.
Удаление безопасно — итоговая сумма Spend не изменится.

### 2.3 Типы данных

Запускаем `.info()` по каждой таблице и тут же фиксируем выводы:
какие колонки нужно конвертировать и почему.

#### Deals

In [ ]:
deals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21595 entries, 0 to 21594
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id                   21593 non-null  object 
 1   Deal Owner Name      21564 non-null  object 
 2   Closing Date         14645 non-null  object 
 3   Quality              19340 non-null  object 
 4   Stage                21593 non-null  object 
 5   Lost Reason          16124 non-null  object 
 6   Page                 21593 non-null  object 
 7   Campaign             16067 non-null  object 
 8   SLA                  15533 non-null  object 
 9   Content              14147 non-null  object 
 10  Term                 12454 non-null  object 
 11  Source               21593 non-null  object 
 12  Payment Type         496 non-null    object 
 13  Product              3592 non-null   object 
 14  Education Type       3300 non-null   object 
 15  Created Time         21593 non-null 

| Колонка | Тип сейчас | Нужный тип | Почему |
|---|---|---|---|
| `Created Time` | object | datetime | Нужен для расчёта длительности сделки и временного ряда |
| `Closing Date` | object | datetime | Нужен для расчёта времени закрытия и проверки аномалий |
| `SLA` | object | int (секунды) | Хранит строку вида `"2h 30m 15s"` — нужно разобрать в число |
| `Initial Amount Paid` | object | float | Содержит символ `€` и пробелы — нужна числовая очистка |
| `Offer Total Amount` | object | float | Та же проблема: `€` и пробелы |

#### Calls

In [ ]:
calls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95874 entries, 0 to 95873
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Id                          95874 non-null  object 
 1   Call Start Time             95874 non-null  object 
 2   Call Owner Name             95874 non-null  object 
 3   CONTACTID                   91941 non-null  object 
 4   Call Type                   95874 non-null  object 
 5   Call Duration (in seconds)  95791 non-null  float64
 6   Call Status                 95874 non-null  object 
 7   Dialled Number              0 non-null      float64
 8   Outgoing Call Status        86875 non-null  object 
 9   Scheduled in CRM            86875 non-null  float64
 10  Tag                         0 non-null      float64
dtypes: float64(4), object(7)
memory usage: 8.0+ MB


| Колонка | Тип сейчас | Нужный тип | Почему |
|---|---|---|---|
| `Call Start Time` | object | datetime | Нужен для временного анализа звонков |

Остальные колонки уже имеют правильные типы: `Call Duration (in seconds)` — числовой,
категориальные (`Call Type`, `Call Status`) — строки, что корректно.

#### Contacts

In [ ]:
contacts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18548 entries, 0 to 18547
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Id                  18548 non-null  object
 1   Contact Owner Name  18548 non-null  object
 2   Created Time        18548 non-null  object
 3   Modified Time       18548 non-null  object
dtypes: object(4)
memory usage: 579.8+ KB


| Колонка | Тип сейчас | Нужный тип | Почему |
|---|---|---|---|
| `Created Time` | object | datetime | Нужен для анализа динамики появления новых контактов |
| `Modified Time` | object | datetime | Нужен для отслеживания активности по контактам |

#### Spend

In [ ]:
spend.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20779 entries, 0 to 20778
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         20779 non-null  datetime64[ns]
 1   Source       20779 non-null  object        
 2   Campaign     14785 non-null  object        
 3   Impressions  20779 non-null  int64         
 4   Spend        20779 non-null  float64       
 5   Clicks       20779 non-null  int64         
 6   AdGroup      13951 non-null  object        
 7   Ad           13951 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 1.3+ MB


| Колонка | Тип сейчас | Нужный тип | Почему |
|---|---|---|---|
| `Date` | object | datetime | Нужен для анализа рекламных расходов по времени |

Числовые колонки (`Impressions`, `Spend`, `Clicks`) — проверим при очистке;
если уже float/int, дополнительных действий не нужно.

### 2.4 Детальный осмотр Deals

В разделе 2.3 `.info()` показал что несколько колонок Deals имеют тип `object`
вместо числового или datetime. Идём смотреть: что именно лежит в этих колонках
и есть ли другие аномалии в данных.

#### SLA — почему object?

In [ ]:
# Смотрим на примеры значений SLA
deals["SLA"].dropna().head(10)

0     4 days, 21:19:46
1             01:48:36
3             13:03:58
4             05:03:32
5             00:28:51
6       1 day, 0:10:21
7             02:25:07
8             00:18:20
9             13:28:43
10      1 day, 4:25:39
Name: SLA, dtype: object

**Что видим:** значения вида `"4 days, 21:19:46"` и `"01:48:36"` — строки с буквами,
не числа. Pandas не смог прочитать их как число → сохранил как object.

**Вывод:** нужен парсер который переведёт в секунды.

#### Initial Amount Paid и Offer Total Amount — почему object?

In [108]:
# Смотрим на примеры денежных колонок
deals[["Initial Amount Paid", "Offer Total Amount"]].dropna().head(8)

,Initial Amount Paid,Offer Total Amount
0,100,"€ 2.900,00"
1,3000,"€ 2.900,00"
2,"€ 3.500,00",3500
4,0,"€ 2.900,00"
5,"€ 3.500,00",3500
6,"€ 3.500,00",3500
7,1500,"€ 2.900,00"
8,"€ 3.500,00",3500


**Что видим:** значения содержат символ `€` и пробелы — pandas не может
прочитать `"€1 500.00"` как число, поэтому тип остался object.

**Вывод:** нужно удалить `€` и пробелы, конвертировать в `float`.

#### Аномалия: Closing Date раньше Created Time?

In [ ]:
# Временно конвертируем даты чтобы проверить аномалию
created = pd.to_datetime(deals["Created Time"], dayfirst=True, errors="coerce")
closing = pd.to_datetime(deals["Closing Date"], dayfirst=True, errors="coerce")
mask_date = (closing < created) & closing.notna()
print(f"Строк где Closing Date < Created Time: {mask_date.sum()}")
deals.loc[mask_date, ["Created Time", "Closing Date"]].head(5)

**Что видим:** сделки где дата закрытия стоит раньше даты создания —
это логически невозможно. Ошибки ручного ввода.

**Вывод:** `Closing Date` в таких строках заменим на `NaT` и поставим флаг `closing_date_error`
(чтобы не потерять остальные данные строки — источник, сумму, менеджера).

#### Аномалия: первый платёж больше суммы предложения?

In [ ]:
# Временно извлекаем числа чтобы проверить аномалию
def parse_amount(s):
    if pd.isna(s): return None
    return pd.to_numeric(
        str(s).replace("€", "").replace(" ", "").replace(",", "."),
        errors="coerce"
    )

iap = deals["Initial Amount Paid"].apply(parse_amount)
ota = deals["Offer Total Amount"].apply(parse_amount)
mask_amt = (iap > ota) & iap.notna() & ota.notna()
print(f"Строк где Initial Amount > Offer Total: {mask_amt.sum()}")
deals.loc[mask_amt, ["Initial Amount Paid", "Offer Total Amount"]].head(5)

**Что видим:** строки где первый платёж больше полной стоимости курса.
Это невозможно по смыслу — часть не может быть больше целого.
По правилам проекта (FAQ): *"Очевидная ошибка. Аналитик принимает решение."*

**Вывод:** `Offer Total Amount` в таких строках → `NaN`.
Доверяем `Initial Amount Paid` как факту платежа, а `Offer Total` как введённому вручную
и ошибочному → обнуляем именно его. Флаг `amount_error`.

#### Level of Deutsch — насколько стандартные значения?

In [ ]:
# Смотрим уникальные значения Level of Deutsch
print(f"Уникальных значений: {deals['Level of Deutsch'].nunique()}")
deals["Level of Deutsch"].value_counts().head(30)

**Что видим:** вместо стандартной шкалы (A1/A2/B1/B2/C1/C2) — сотни вариантов
написания: `"б1"`, `"B 1"`, `"b2"`, `"в1"`, `"Б2"`, `"beginner"` и т.д.
Менеджеры заполняли поле вручную без валидации.

**Вывод:** нужна regex-нормализация → привести всё к шкале A0/A1/A2/B1/B2/C1/C2.

#### City, Quality, Education Type — неожиданные значения?

Категориальные колонки тоже заполнялись вручную. Смотрим что реально в них лежит.

In [109]:
# Quality
print("=== Quality ===")
print(deals["Quality"].value_counts())


=== Quality ===
Quality
E - Non Qualified    7634
D - Non Target       6248
C - Low              3459
B - Medium           1564
A - High              432
F                       3
Name: count, dtype: int64


In [110]:
# Education Type
print("=== Education Type ===")
print(deals["Education Type"].value_counts())


=== Education Type ===
Education Type
Morning    2895
Evening     404
#REF!         1
Name: count, dtype: int64


In [111]:
# City — топ значений
print("=== City (топ-20) ===")
print(deals["City"].value_counts().head(20))


=== City (топ-20) ===
City
-                         348
Berlin                    182
München                    74
Hamburg                    62
Nürnberg                   45
Leipzig                    45
Düsseldorf                 33
Dresden                    28
Frankfurt                  27
Dortmund                   26
Köln                       25
Stuttgart                  20
Duisburg                   19
Hannover                   19
Bremen                     17
Karlsruhe                  16
Essen                      16
Bochum                     15
Villingen-Schwenningen     14
Augsburg                   13
Name: count, dtype: int64


**Что видим:** среди значений встречаются `-`, `"null"`, цифры, случайные строки —
это не реальные города или категории, а артефакты ручного ввода.

**Вывод:** такие значения заменим на `NaN` — они не несут аналитической ценности.

### 2.5 Итоговый план очистки

На основании находок выше (2.1 – 2.4) формируем конкретный план:

| # | Таблица | Что нашли | Что делаем |
|---|---|---|---|
| 1 | **Calls** | `Dialled Number`, `Tag` — 100% пусты | Удаляем колонки |
| 2 | **Calls** | `Call Start Time` — строка | Конвертируем в `datetime` |
| 3 | **Contacts** | `Created Time`, `Modified Time` — строки | Конвертируем в `datetime` |
| 4 | **Spend** | 917 полных дублей, расходы в них = 0 | Удаляем дубли |
| 5 | **Spend** | `Date` — строка | Конвертируем в `datetime` |
| 6 | **Deals** | 1 дубль по `Id` (полных дублей нет) | Разбираем в разделе 4.1 |
| 7 | **Deals** | `Created Time`, `Closing Date` — строки | Конвертируем в `datetime` |
| 8 | **Deals** | `SLA` — строка вида `"Xh Ym Zs"` | Парсим в секунды (`int`) |
| 9 | **Deals** | `Initial Amount Paid`, `Offer Total` — строки с €| Чистим, конвертируем в `float` |
| 10 | **Deals** | 44 строки: `Closing Date` < `Created Time` | Заменяем Closing Date на `NaT` + флаг |
| 11 | **Deals** | 58 строк: `Initial Amount` > `Offer Total` | Заменяем `Offer Total` на `NaN` + флаг |
| 12 | **Deals** | `Level of Deutsch` — произвольный текст | Regex-нормализация → шкала A0–C2 |
| 13 | **Deals** | `City`, `Quality`, `Education Type` — мусорные значения | Заменяем на `NaN` |

Deals выносим в отдельный Этап 4 — у неё больше всего исправлений (пп. 6–13).

## 3. Очистка Calls, Contacts, Spend

### 3.1 Очистка Calls

По заданию нужно удалить неактуальные столбцы. Проверим кандидатов:
по обзору данных колонки `Dialled Number` и `Tag` выглядели пустыми.

In [ ]:
# Проверяем, сколько значений реально заполнено в подозрительных колонках
total = len(calls)
print(f'Dialled Number: пропусков {calls["Dialled Number"].isna().sum()} из {total}')
print(f'Tag:            пропусков {calls["Tag"].isna().sum()} из {total}')
print('Дубликатов по Id:', calls['Id'].duplicated().sum())

Dialled Number: пропусков 95874 из 95874
Tag:            пропусков 95874 из 95874
Дубликатов по Id: 0


Обе колонки пусты на 100% — информации в них нет, удаляем.
Дубликатов нет. Дальше приводим типы: дату звонка — в `datetime`,
длительность — в число.

In [ ]:
# Удаляем полностью пустые (неактуальные) колонки
calls = calls.drop(columns=['Dialled Number', 'Tag'])

In [ ]:
# Приводим типы: дата звонка и длительность в секундах
calls['Call Start Time'] = pd.to_datetime(
    calls['Call Start Time'], format='%d.%m.%Y %H:%M'
)
calls['Call Duration (in seconds)'] = pd.to_numeric(
    calls['Call Duration (in seconds)'], errors='coerce'
)

# Контроль типов
calls.dtypes

Id                                    object
Call Start Time               datetime64[ns]
Call Owner Name                       object
CONTACTID                             object
Call Type                             object
Call Duration (in seconds)           float64
Call Status                           object
Outgoing Call Status                  object
Scheduled in CRM                     float64
dtype: object

### 3.2 Очистка Contacts

Начинаем с самой простой таблицы. Проверим её на дубликаты и пропуски.

In [ ]:
# Проверяем качество таблицы контактов
print('Дубликатов по Id:', contacts['Id'].duplicated().sum())
print('Пропусков всего:', contacts.isna().sum().sum())

Дубликатов по Id: 0
Пропусков всего: 0


Таблица чистая. Единственная проблема — даты хранятся строками вида
`27.06.2023 11:28`. Приводим их к типу `datetime`, указывая формат явно
(день.месяц.год — иначе pandas может перепутать день с месяцем).

In [ ]:
# Преобразуем обе колонки дат из строк в datetime
for col in ['Created Time', 'Modified Time']:
    contacts[col] = pd.to_datetime(contacts[col], format='%d.%m.%Y %H:%M')

# Контроль: смотрим типы после преобразования
contacts.dtypes

Id                            object
Contact Owner Name            object
Created Time          datetime64[ns]
Modified Time         datetime64[ns]
dtype: object

### 3.3 Очистка Spend

В таблице расходов есть полные дубликаты строк. Удалять их вслепую
опасно: если в копиях есть деньги, мы занизим расходы и исказим расчёт
стоимости привлечения (CAC) в юнит-экономике.

Поэтому сначала проверяем, что именно дублируется.

In [ ]:
# Выделяем лишние копии (все вхождения дубликата, кроме первого)
extra_copies = spend[spend.duplicated()]
display(extra_copies)

print('Лишних копий-дубликатов:', len(extra_copies))
print('Сумма Spend в лишних копиях:      ', extra_copies['Spend'].sum())
print('Сумма Impressions в лишних копиях:', extra_copies['Impressions'].sum())
print('Сумма Clicks в лишних копиях:     ', extra_copies['Clicks'].sum())

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
2461,2023-07-23,Bloggers,NaN,0,0.0,0,NaN,NaN
2490,2023-07-24,Bloggers,NaN,0,0.0,0,NaN,NaN
2559,2023-07-25,Bloggers,NaN,0,0.0,0,NaN,NaN
2626,2023-07-26,Bloggers,NaN,0,0.0,0,NaN,NaN
2655,2023-07-27,Bloggers,NaN,0,0.0,0,NaN,NaN
...,...,...,...,...,...,...,...,...
20750,2024-06-21,Bloggers,NaN,0,0.0,0,NaN,NaN
20754,2024-06-21,Facebook Ads,NaN,0,0.0,0,NaN,NaN
20763,2024-06-21,SMM,NaN,0,0.0,0,NaN,NaN
20764,2024-06-21,Telegram posts,NaN,0,0.0,0,NaN,NaN


Лишних копий-дубликатов: 917
Сумма Spend в лишних копиях:       0.0
Сумма Impressions в лишних копиях: 0
Сумма Clicks в лишних копиях:      46


**Проверка показала:** в лишних копиях нет ни расходов, ни показов
(это в основном строки канала Bloggers без кампании, забитые нулями).
Удаление безопасно — но на всякий случай контролируем сумму расходов
до и после.

In [ ]:
# Запоминаем контрольную сумму, удаляем дубликаты, сверяем
spend_total_before = spend['Spend'].sum()

spend = spend.drop_duplicates().reset_index(drop=True)

print(f'Было строк: {rows_before["spend"]}, стало: {len(spend)}')
print('Сумма Spend не изменилась:',
      spend['Spend'].sum() == spend_total_before)

Было строк: 20779, стало: 19862
Сумма Spend не изменилась: True


Остались пропуски в `Campaign` / `AdGroup` / `Ad` — это норма для
каналов без рекламных кампаний (Organic, SMM, Webinar и т.п.), не трогаем.
Колонка `Date` уже имеет тип `datetime`, числовые колонки корректны.

In [ ]:
# Контроль: пропуски и типы после очистки
print(spend.isna().sum())
spend.dtypes

Date              0
Source            0
Campaign       5077
Impressions       0
Spend             0
Clicks            0
AdGroup        5911
Ad             5911
dtype: int64


Date           datetime64[ns]
Source                 object
Campaign               object
Impressions             int64
Spend                 float64
Clicks                  int64
AdGroup                object
Ad                     object
dtype: object

## 4. Очистка Deals

### 4.1 Битые строки и дубликаты

Проверяем строки без Id и дубликаты (Id мы читали строками, поэтому
проверка корректна).

In [ ]:
# Проверяем целостность таблицы сделок
print('Строк без Id:', deals['Id'].isna().sum())
print('Полных дубликатов строк:', deals.duplicated().sum())
print('Дубликатов по Id:',
      deals.dropna(subset=['Id'])['Id'].duplicated().sum())

Строк без Id: 2
Полных дубликатов строк: 0
Дубликатов по Id: 0


Дубликатов нет. Две строки без Id — мусор выгрузки (одна полностью
пустая, во второй заполнено одно поле) — удаляем.

In [ ]:
# Удаляем битые строки без идентификатора
deals = deals.dropna(subset=['Id']).reset_index(drop=True)
print('Стало строк:', len(deals))

Стало строк: 21593


### 4.2 Даты

`Created Time` и `Closing Date` хранятся строками — приводим к `datetime`.
Формат указываем явно: день.месяц.год.

In [ ]:
# Дата создания содержит время, дата закрытия — только день
deals['Created Time'] = pd.to_datetime(
    deals['Created Time'], format='%d.%m.%Y %H:%M'
)
deals['Closing Date'] = pd.to_datetime(deals['Closing Date'], dayfirst=True)

# Контроль: диапазоны дат после преобразования
deals[['Created Time', 'Closing Date']].agg(['min', 'max'])

,Created Time,Closing Date
min,2023-07-03 17:03:00,2022-10-11
max,2024-06-21 15:30:00,2024-12-11


### 4.3 SLA → секунды

`SLA` — время ответа менеджера на заявку. Excel хранит его неоднородно:
как время (`чч:мм:сс`) или как интервал (`timedelta`, если ответ занял
больше суток). Для анализа удобнее одно число — переводим всё в секунды.

In [ ]:
def sla_to_seconds(value):
    """Перевести SLA в секунды.

    Поддерживает три варианта хранения: datetime.time, timedelta
    и строку 'чч:мм:сс'. Пропуски остаются NaN.
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, dt.timedelta):
        return value.total_seconds()
    if isinstance(value, dt.time):
        return value.hour * 3600 + value.minute * 60 + value.second
    try:
        hours, minutes, seconds = map(int, str(value).split(':'))
        return hours * 3600 + minutes * 60 + seconds
    except ValueError:
        return np.nan

In [ ]:
# Применяем функцию и заменяем исходную колонку числовой версией
deals['SLA (seconds)'] = deals['SLA'].apply(sla_to_seconds)
deals = deals.drop(columns=['SLA'])

# Контроль: статистика SLA в секундах
deals['SLA (seconds)'].describe().round(0)

count       15533.0
mean       115826.0
std        737253.0
min             3.0
25%          4380.0
50%         19894.0
75%         56318.0
max      26908464.0
Name: SLA (seconds), dtype: float64

### 4.4 Денежные колонки

`Initial Amount Paid` (первый платёж) и `Offer Total Amount` (сумма
предложения) почти все числовые, но встречаются значения в европейском
формате с символом валюты: `€ 3.500,00` (точка — разделитель тысяч,
запятая — десятичный разделитель). Такие значения pandas прочитал как
текст. Пишем парсер.

In [ ]:
def parse_money(value):
    """Привести денежное значение к float.

    Числа возвращаются как есть. Строки вида '€ 3.500,00'
    преобразуются: убираем €, точки тысяч, запятую меняем на точку.
    Нераспознанные значения -> NaN.
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value).replace('€', '').replace(' ', '')
    text = text.replace('.', '').replace(',', '.')
    try:
        return float(text)
    except ValueError:
        return np.nan

In [ ]:
# Применяем парсер к обеим денежным колонкам
for col in ['Initial Amount Paid', 'Offer Total Amount']:
    deals[col] = deals[col].apply(parse_money)

# Контроль: сводная статистика по деньгам
deals[['Initial Amount Paid', 'Offer Total Amount']].describe().round(1)

,Initial Amount Paid,Offer Total Amount
count,4165.0,4185.0
mean,950.1,7128.7
std,1422.2,4636.0
min,0.0,0.0
25%,300.0,3000.0
50%,1000.0,11000.0
75%,1000.0,11000.0
max,11500.0,11500.0


### 4.5 Флаг демо-доступов

По условиям проекта оплаты на суммы **0, 1 и 9 — это демо-доступы за
символическую плату**. Это не ошибка данных, но при расчёте среднего
чека такие «оплаты» исказят картину. Помечаем их флагом, чтобы в
анализе легко включать/исключать.

In [ ]:
# Флаг демо-доступа по условиям проекта (суммы 0, 1, 9)
deals['Is Demo'] = deals['Initial Amount Paid'].isin([0, 1, 9])
print('Демо-сделок:', deals['Is Demo'].sum())

Демо-сделок: 880


### 4.6 Аномалия: дата закрытия раньше даты создания

По условиям проекта это **очевидная ошибка**, способ исправления — на
усмотрение аналитика. Сначала оценим масштаб и посмотрим, какие это
сделки.

In [ ]:
# Ищем строки, где сделка «закрылась» раньше, чем была создана
bad_close = deals['Closing Date'].dt.date < deals['Created Time'].dt.date

print('Строк с Closing Date < Created Time:', bad_close.sum())
print()
print('Стадии этих сделок:')
print(deals.loc[bad_close, 'Stage'].value_counts())

Строк с Closing Date < Created Time: 44

Стадии этих сделок:
Stage
Lost            40
Payment Done     4
Name: count, dtype: int64


**Решение:** дату закрытия в этих 44 строках считаем недостоверной —
заменяем на `NaT` и помечаем флагом `Closing Date Error`. Так ошибочные
даты не исказят анализ длительности сделок, а сами сделки (стадия,
суммы, источник) останутся в данных.

In [ ]:
# Помечаем ошибочные строки флагом и обнуляем недостоверную дату
deals['Closing Date Error'] = bad_close
deals.loc[bad_close, 'Closing Date'] = pd.NaT

### 4.7 Аномалия: первый платёж больше суммы предложения

`Initial Amount Paid > Offer Total Amount` — по условиям проекта тоже очевидная
ошибка ручного ввода менеджеров.

**Решение:** первый платёж — это фактически полученные деньги, ему
доверяем больше. Сумму предложения в таких строках считаем
недостоверной → `NaN`, строки помечаем флагом `Amount Error`.

In [ ]:
# Находим строки с противоречием в суммах
amount_error = deals['Initial Amount Paid'] > deals['Offer Total Amount']
print('Строк с IAP > OTA:', amount_error.sum())

# Помечаем флагом и убираем недостоверную сумму предложения
deals['Amount Error'] = amount_error.fillna(False)
deals.loc[deals['Amount Error'], 'Offer Total Amount'] = np.nan

Строк с IAP > OTA: 58


### 4.8 Уровень немецкого языка

`Level of Deutsch` менеджеры заполняли вручную в свободной форме:
«б1 (ждет серт)», «А2 ( Б1 март )», «25 лет живет в Германии»,
встречаются даже почтовые адреса. Уникальных значений — сотни.

**Решение:** извлекаем регулярным выражением **первый упомянутый
уровень** по стандартной шкале A0–C2:
- кириллические буквы приводим к латинице («б1» и «в1» обе означают B1:
  первая фонетически, вторая визуально);
- записи «нет», «не учил», «никакой», «нулевой» считаем уровнем `A0`;
- всё, из чего уровень извлечь нельзя, — `NaN`.

Это сознательное огрубление (например, «ждёт результата B1» засчитается
как B1), но единая шкала для анализа важнее точности единичных записей.

In [ ]:
def extract_deutsch_level(value):
    """Извлечь уровень немецкого (A0–C2) из свободного текста.

    Возвращает стандартный уровень или NaN, если распознать нельзя.
    """
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()

    # Явное отсутствие языка считаем уровнем A0
    if re.search(r'нет|не учил|никак|нулев|ня-0|ня - 0', text):
        return 'A0'

    # Кириллица -> латиница (б и в обе означают B)
    text = text.translate(str.maketrans({'а': 'a', 'б': 'b',
                                         'в': 'b', 'с': 'c'}))

    # Ищем первое сочетание буквы уровня и цифры, например 'b1'
    match = re.search(r'([abc])\s*([0-2])', text)
    if match:
        level = match.group(1).upper() + match.group(2)
        if level in ('A0', 'A1', 'A2', 'B1', 'B2', 'C1', 'C2'):
            return level
    return np.nan

In [ ]:
# Применяем функцию: создаём новую колонку со стандартной шкалой
deals['Deutsch Level'] = deals['Level of Deutsch'].apply(
    extract_deutsch_level
)

# Контроль: сколько записей удалось привести к шкале
print('Заполнено в исходной колонке:', deals['Level of Deutsch'].notna().sum())
print('Извлечено уровней:           ', deals['Deutsch Level'].notna().sum())

Заполнено в исходной колонке: 1251
Извлечено уровней:            1204


In [ ]:
# Смотрим итоговое распределение по уровням
deals['Deutsch Level'].value_counts().sort_index()

Deutsch Level
A0     17
A1     25
A2    149
B1    814
B2    170
C1     26
C2      3
Name: count, dtype: int64

### 4.9 Категориальные колонки: City, Education Type, Quality

Точечные проблемы, найденные при обзоре данных:
- `City`: значение «-» — это отсутствие данных → `NaN`; убираем лишние
  пробелы. Глубокую унификацию городов (где вместо города вписан адрес)
  отложим до географического анализа;
- `Education Type`: значение `#REF!` — ошибка экспорта из электронных
  таблиц → `NaN`;
- `Quality`: шкала качества лида A–E; значение «F» (3 строки, все Lost)
  в шкале проекта не описано → считаем мусорным, `NaN`.

In [ ]:
# Чистим три категориальные колонки по правилам выше
deals['City'] = (
    deals['City']
    .astype('string')
    .str.strip()
    .replace({'-': pd.NA, '': pd.NA})
)
deals['Education Type'] = deals['Education Type'].replace('#REF!', np.nan)
deals['Quality'] = deals['Quality'].replace('F', np.nan)

In [ ]:
# Контроль: распределения после очистки
print('City заполнено:', deals['City'].notna().sum())
print()
print(deals['Education Type'].value_counts(dropna=False).head())
print()
print(deals['Quality'].value_counts(dropna=False))

City заполнено: 2163

Education Type
NaN        18294
Morning     2895
Evening      404
Name: count, dtype: int64

Quality
E - Non Qualified    7634
D - Non Target       6248
C - Low              3459
NaN                  2256
B - Medium           1564
A - High              432
Name: count, dtype: int64


### 4.10 Числовые колонки длительности обучения

`Course duration` (длительность курса) и `Months of study` (сколько
месяцев студент уже отучился) приводим к числам.

In [ ]:
# Приводим длительности к числовому типу
deals['Course duration'] = pd.to_numeric(
    deals['Course duration'], errors='coerce'
)
deals['Months of study'] = pd.to_numeric(
    deals['Months of study'], errors='coerce'
)

# Контроль: сводная статистика
deals[['Course duration', 'Months of study']].describe().round(2)

,Course duration,Months of study
count,3587.00,840.00
mean,10.20,5.44
std,1.83,2.92
min,6.00,0.00
25%,11.00,3.00
50%,11.00,5.00
75%,11.00,8.00
max,11.00,11.00


## 5. Итоги очистки и сохранение

Собираем таблицу «было/стало» — она наглядно показывает объём
проделанной работы и пригодится в отчёте.

In [ ]:
# Итоговая таблица «было/стало» по всем четырём датасетам
summary = pd.DataFrame({
    'Строк было': rows_before,
    'Строк стало': {
        'deals': len(deals),
        'calls': len(calls),
        'contacts': len(contacts),
        'spend': len(spend),
    },
})
summary['Удалено строк'] = summary['Строк было'] - summary['Строк стало']
summary['Что сделано'] = [
    '2 битые строки; даты; SLA -> сек; парсинг €; флаги ошибок; Deutsch Level',
    'удалены пустые колонки Dialled Number и Tag; даты',
    'даты',
    'удалено 917 нулевых дубликатов (суммы не изменились)',
]
summary

,Строк было,Строк стало,Удалено строк,Что сделано
deals,21595,21593,2,2 битые строки; даты; SLA -> сек; парсинг €; ф...
calls,95874,95874,0,удалены пустые колонки Dialled Number и Tag; даты
contacts,18548,18548,0,даты
spend,20779,19862,917,удалено 917 нулевых дубликатов (суммы не измен...


In [ ]:
# Сохраняем чистые датасеты в data_clean/
# Важно: при последующем чтении этих CSV снова указывать dtype=str для Id!
deals.to_csv(CLEAN_DIR / 'deals_clean.csv', index=False)
calls.to_csv(CLEAN_DIR / 'calls_clean.csv', index=False)
contacts.to_csv(CLEAN_DIR / 'contacts_clean.csv', index=False)
spend.to_csv(CLEAN_DIR / 'spend_clean.csv', index=False)

In [ ]:
# Контроль: файлы созданы, размеры адекватные
for path in sorted(CLEAN_DIR.glob('*.csv')):
    size_mb = path.stat().st_size / 1024 / 1024
    print(f'{path.name} - {size_mb:.2f} МБ')

calls_clean.csv - 10.58 МБ
contacts_clean.csv - 1.31 МБ
deals_clean.csv - 4.31 МБ
spend_clean.csv - 1.23 МБ


## Выводы этапа

1. **Id только строками** — `float64` теряет точность 19-значных
   идентификаторов, появляются ложные дубликаты.
2. **Contacts**: таблица чистая, приведены даты.
3. **Calls**: удалены 2 полностью пустые колонки, приведены даты и
   длительность.
4. **Spend**: удалено 917 дубликатов — перед удалением проверено, что
   все копии нулевые; суммы расходов не изменились.
5. **Deals**: удалены 2 битые строки; даты и суммы приведены к
   корректным типам (включая €-формат); аномалии помечены флагами и
   нейтрализованы по правилам проекта (закрытие раньше создания — 44 строки,
   платёж больше предложения — 58 строк); свободный текст уровня
   немецкого сведён к шкале A0–C2 (распознано 1204 из 1251 записей);
   категории очищены от мусора («-», `#REF!`, «F»).
6. **Демо-доступы** (оплаты 0/1/9, всего 880) помечены флагом `Is Demo`.
7. Чистые данные сохранены в `data_clean/` — все следующие этапы
   работают только с ними.

**Следующий этап:** `02_descriptive.ipynb` — описательная статистика.